<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH

http://tpq.io | [training@tpq.io](mailto:trainin@tpq.io) | [@dyjh](http://twitter.com/dyjh)

## Infrastructure

Using `conda` as the environment and package manager:

https://conda.io/en/master/miniconda.html

With `conda` it is easy to manage Python environments:

https://conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html

## Financial Data

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_practice.git
import sys
sys.path.append('python_for_algo_trading_practice')


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
url = 'https://certificate.tpq.io/findata.csv'

In [ ]:
raw = pd.read_csv(url, index_col=0, parse_dates=True)

In [ ]:
print(raw)

In [ ]:
raw.info()

In [ ]:
from pylab import plt
plt.style.use('seaborn-v0_8')
%config InlineBackend.figure_format = 'svg'

In [ ]:
symbol = 'EUR='

In [ ]:
data = pd.DataFrame(raw[symbol])

In [ ]:
data.plot();

In [ ]:
data['r'] = np.log(data / data.shift(1))

In [ ]:
data.head()

## Strategy Code 

In [ ]:
lags = 7  # number of lags to be used

In [ ]:
cols = list()
for lag in range(1, lags + 1):
    col = f'lag_{lag}'
    data[col] = data['r'].shift(lag)  # lags are created
    cols.append(col)

In [ ]:
cols  # column names for the lagged data

In [ ]:
print(data.head(8))

In [ ]:
data.dropna(inplace=True)  # deleting rows with NaNs

In [ ]:
reg = np.linalg.lstsq(data[cols], data['r'], rcond=-1)[0]  # OLS regression step
reg

In [ ]:
np.dot(data[cols], reg)  # prediction/estimation step

In [ ]:
data['e'] = np.dot(data[cols], reg)  # prediction/estimation step

In [ ]:
data[['r', 'e']].plot();

In [ ]:
data['p'] = np.sign(data['e'])
# transformation of estimates to positional/directional values (+1 or -1)

In [ ]:
data.head()

In [ ]:
data['p'].iloc[-25:].plot();

In [ ]:
data['p'].diff()

In [ ]:
(data['p'].diff() != 0)

In [ ]:
sum(data['p'].diff() != 0)

## Backtesting Code

All of the following is calculated under simplifying assumptions, like assuming zero transaction costs and no collateral/margin for short positions.

In [ ]:
data['s'] = data['p'] * data['r']

In [ ]:
data[['r', 's']].sum()  # sum of the log returns (benchmark vs. strategy)

In [ ]:
data[['r', 's']].sum().apply(np.exp)  # gross return (benchmark vs. strategy)

In [ ]:
data[['r', 's']].cumsum().apply(np.exp).plot();  # gross return over time (benchmark vs. strategy)

## Train-Test Split

In [ ]:
split = int(len(data) * 0.8)
split

In [ ]:
train = data.iloc[:split].copy()

In [ ]:
test = data.iloc[split:].copy()

In [ ]:
reg = np.linalg.lstsq(train[cols], train['r'], rcond=-1)[0]
reg

In [ ]:
test['e'] = np.dot(test[cols], reg)

In [ ]:
test['p'] = np.sign(test['e'])

In [ ]:
test['s'] = test['p'] * test['r']

In [ ]:
test[['r', 's']].sum().apply(np.exp)

In [ ]:
test[['r', 's']].cumsum().apply(np.exp).plot();

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>